# 数据探索 — 莱势明 14 张业务表深度分析

本 notebook 演示如何用 Python 对配套数据集（14 张表，57 万行）进行：
- 描述统计
- 相关性分析
- 时间序列分解
- 异常事件挖掘

**配套数据集**：见 `datasets/` 目录


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams["font.sans-serif"] = ["PingFang SC", "SimHei"]

## 1. 加载全部 14 张表

In [ ]:
import os
DATA_DIR = "../datasets"
tables = {}
for f in os.listdir(DATA_DIR):
    if f.endswith(".csv"):
        df = pd.read_csv(os.path.join(DATA_DIR, f))
        tables[f.replace(".csv", "")] = df
        print(f"{f:30s}  shape={df.shape}")
print(f"\n共 {sum(len(v) for v in tables.values()):,} 行数据")

## 2. 工单数据描述统计

In [ ]:
wo = tables["work_orders"]
print("订单数量分布:")
print(wo["qty"].describe())
print("\n优先级分布:")
print(wo["priority"].value_counts(normalize=True))
print("\n机器类型分布:")
print(wo["machine_type"].value_counts())

## 3. 质量检验异常挖掘

In [ ]:
qi = tables["quality_inspections"]
print(f"总检验次数: {len(qi):,}")
print(f"PASS 率: {(qi['result']=='PASS').mean()*100:.2f}%")
print(f"\n缺陷类型 TOP 5:")
top_defects = qi[qi["main_defect_type"] != ""]["main_defect_type"].value_counts().head(5)
print(top_defects)

fig, ax = plt.subplots(figsize=(10, 4))
top_defects.plot(kind="bar", ax=ax, color="#C00000")
ax.set_ylabel("发生次数"); ax.set_title("缺陷类型 TOP 5")
plt.tight_layout(); plt.show()

## 4. 设备能耗趋势

In [ ]:
energy = tables["energy_logs"]
energy["log_date"] = pd.to_datetime(energy["log_date"])
daily_energy = energy.groupby("log_date")["energy_consumed_kwh"].sum()

fig, ax = plt.subplots(figsize=(12, 4))
daily_energy.rolling(7).mean().plot(ax=ax, color="#385723", linewidth=2)
ax.set_title("全厂日能耗趋势 (7 天移动平均)")
ax.set_ylabel("kWh"); plt.tight_layout(); plt.show()

## 5. 换模时间统计

In [ ]:
co = tables["changeovers"]
print(f"总换模次数: {len(co):,}")
print(f"同家族换模 (短):    {(co['family_from']==co['family_to']).sum():,}  avg={co[co['family_from']==co['family_to']]['actual_minutes'].mean():.1f} 分钟")
print(f"跨家族换模 (长):    {(co['family_from']!=co['family_to']).sum():,}  avg={co[co['family_from']!=co['family_to']]['actual_minutes'].mean():.1f} 分钟")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(co["actual_minutes"], bins=50, color="#0070C0", alpha=0.7)
ax.set_xlabel("实际换模时间 (分钟)"); ax.set_ylabel("频数")
ax.set_title("换模时间分布"); plt.tight_layout(); plt.show()

## 总结
本数据集涵盖：
- **生产**：工单、设备日志、换模、班次
- **质量**：质量检验、缺陷类型
- **能耗**：每班次每机器能耗记录
- **物料**：物料消耗与损耗
- **维修**：设备故障与维修事件
- **库存**：库存日快照

可支持的研究方向：
1. 加工时长预测（回归任务）
2. 缺陷率预测（分类任务）
3. 设备故障预测（生存分析）
4. 排产优化（组合优化）
5. 能耗优化（多目标优化）
6. 库存优化（动态批量）
